1. Configs

In [ ]:
import sys
import os

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

from config import Config
from pipeline.context import ClaimContext
from pipeline.output import output
from pipeline.pipeline_factory import pipeline_rule_verdict, pipeline_majority_verdict
from evaluation.evaluate_predictions import evaluate

/home/systemp2w/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2. Load dataset

In [2]:
import json
from pathlib import Path

def load_averitec_split(split="dev"):
    path = Path("../datasets/averitec") / f"{split}.json"
    
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    return data

3. Run pipeline

In [3]:
data = load_averitec_split("dev")[:1] 

pipeline = pipeline_rule_verdict()

results = []

for item in data:
    context = ClaimContext(
        claim_id=item["id"] if "id" in item else None,
        claim_text=item["claim"],
        claim_date=item.get("claim_date")
    )
    
    result = pipeline.run(context)

    results.append({
    "claim": item["claim"],
    "prediction": result.verdict,
    "gold_label": item.get("label"),

    "pipeline": {
        "questions": getattr(result, "questions", []),
        "evidence": result.evidence,
        "stances": result.stances
    }
})

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 768.85it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/systemp2w/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU1 NVIDIA GeForce GTX 1050 Ti which is of cuda capability 6.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.5) - (12.0)
    
  queued_call()
/home/systemp2w/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pyt

4. Save results

In [4]:
import json

with open("results_dev.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

6. Ground truth compare

In [5]:
correct = 0

for item, pred in zip(data, results):
    if item["label"].lower() == pred["prediction"].lower():
        correct += 1

print("Accuracy:", correct / len(results))

Accuracy: 0.0


In [6]:
evaluate("results_dev.json", "../datasets/averitec/dev.json")


===== Evaluation =====

Accuracy: 0.0

Per-class metrics:

refuted
  Precision: 0
  Recall:    0.0
  F1:        0

Macro F1: 0.0
